In [1]:
import pandas as pd

reviews = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\order_reviews_cleaned.csv"
)

orders = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\orders_cleaned.csv"
)

print(reviews.shape)
print(orders.shape)

(99224, 7)
(99441, 8)


In [3]:
orders = pd.read_csv(
    r"C:\Users\saran\guvi project\cart2insights\data\cleaned\orders_cleaned.csv"
)

print(orders.shape)

(99441, 8)


In [4]:
orders[['order_id',
        'order_purchase_timestamp',
        'order_delivered_customer_date',
        'order_estimated_delivery_date']].head()

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26


In [6]:
date_columns = [
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

In [8]:
orders['delivery_status'] = orders.apply(
    lambda row: 'Delayed'
    if row['order_delivered_customer_date'] > row['order_estimated_delivery_date']
    else 'On Time',
    axis=1
)

In [9]:
orders['delivery_status'].value_counts()

delivery_status
On Time    91614
Delayed     7827
Name: count, dtype: int64

In [11]:
orders['delivery_status'] = None

valid_delivery = (
    orders['order_delivered_customer_date'].notna() &
    orders['order_estimated_delivery_date'].notna()
)

orders.loc[
    valid_delivery &
    (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']),
    'delivery_status'
] = 'Delayed'

orders.loc[
    valid_delivery &
    (orders['order_delivered_customer_date'] <= orders['order_estimated_delivery_date']),
    'delivery_status'
] = 'On Time'

In [13]:
orders['delivery_status'].value_counts(dropna=False)

delivery_status
On Time    88649
Delayed     7827
None        2965
Name: count, dtype: int64

In [14]:
review_delivery = reviews.merge(
    orders[['order_id', 'delivery_status']],
    on='order_id',
    how='inner'
)

print(review_delivery.shape)

(99224, 8)


In [15]:
print(review_delivery['delivery_status'].value_counts(dropna=False))

delivery_status
On Time    88658
Delayed     7701
None        2865
Name: count, dtype: int64


In [16]:
test_data = review_delivery[
    review_delivery['delivery_status'].isin(['On Time', 'Delayed'])
].copy()

on_time_scores = test_data.loc[
    test_data['delivery_status'] == 'On Time',
    'review_score'
].dropna()

delayed_scores = test_data.loc[
    test_data['delivery_status'] == 'Delayed',
    'review_score'
].dropna()

print("On-time reviews:", len(on_time_scores))
print("Delayed reviews:", len(delayed_scores))

On-time reviews: 88658
Delayed reviews: 7701


In [17]:
print("Average On-time review score:", on_time_scores.mean())
print("Average Delayed review score:", delayed_scores.mean())

Average On-time review score: 4.293577567732184
Average Delayed review score: 2.566549798727438
